# 99.7h Stability: When Do Numerical Errors Grow?

## Contents

**Tools:** NumPy; Matplotlib; ipywidgets; SciPy matrix exponential.

**Python used:** Imports, functions, loops, array operations, complex numbers, matrix multiplication, eigenvalue calculations and interactive plots.

**Introduces:** Numerical stability, perturbation growth, amplification factors, step-size restrictions, complex eigenvalues, stability regions and the distinction between stability and accuracy.

1. From Taylor expansion to explicit Euler
2. Tracking differences between closely started numerical solutions
3. Amplification factors and the effect of step size
4. Complex-valued solutions: growth, decay and oscillations
5. Stability versus accuracy
6. Stability of coupled ODE systems
7. Complex eigenvalues and stability regions
8. Explicit and implicit Euler for a harmonic oscillator
9. Practical takeaways for choosing a method and step size

Small differences in initial conditions can shrink or grow during numerical integration. We derive how explicit Euler propagates these differences, explore stability, and extend the analysis to coupled systems and oscillations. Comparing explicit and implicit Euler shows why a stable numerical solution is not necessarily an accurate one.

## Numerical stability

To understand stability, let's revisit how we solve an ODE numerically.

## 1. How do we solve a general ODE numerically?

Consider an initial-value problem

$$\boxed{x'(t)=f(t,x(t)),\qquad x(t_0)=x_0.}$$

We do not know the exact solution $x(t)$. However, we can use the **Taylor expansion** to approximate it as

$$x(t_{i+1})=x(t_i)+h x'(t_i)+\frac{h^2}{2}x''(t_i)+\cdots$$

where $t_i$ is the current time and $t_{i+1}=t_i+h$ is the time after a time step of $h>0$. By discarding the higher-order terms, we obtain

$$x(t_{i+1})\approx x(t_i)+h x'(t_i)+\underbrace{\frac{h^2}{2}x''(t_i)+\cdots}_{\text{discarded terms}}$$

Then we can write an approximation for the first-order derivative term ($x_i\approx x(t_i)$)

$$x'(t_i)\approx\frac{x_{i+1}-x_i}{h}$$

We know from our ODE that $x'(t)=f(t,x(t))$. Replacing with this, we obtain **explicit Euler**:

$$\boxed{x_{i+1}=x_i+h f(t_i,x_i)}$$

## 2. Choose a simple linear function

Now assume the function is linear. 

$$\boxed{f(t,x)=\lambda x,\qquad x'=\lambda x.}$$

Explicit Euler gives the **numerical** update

$$\boxed{x_{i+1}=x_i+h\lambda x_i=(1+h\lambda)x_i.}$$

Apply this repeatedly from $t_0$ to $t_n$:

$$
\begin{gathered}
x_1 = (1+h\lambda)x_0 \\[6pt]
x_2 = (1+h\lambda)x_1 = (1+h\lambda)^2x_0 \\[6pt]
\vdots \\[6pt]
\boxed{x_n = (1+h\lambda)^n x_0}
\end{gathered}
$$


## 3. What do we mean by numerical stability?

Now return to the general question: **Consider an ODE $x'=f(t,x)$. Run the same numerical method twice with slightly different initial values, $x_0$ and $\widetilde{x}_0$. Do the resulting approximations stay close?**

That is the central definition of numerical stability:

> A numerical method is stable if small changes in initial data produce correspondingly small changes in subsequent numerical approximations.

Let

$$\delta_0=x_0-\widetilde{x}_0,\qquad \delta_n=x_n-\widetilde{x}_n.$$

**How is the difference between two solutions affected from the difference in the initial conditions?**

## 4. Subtract the two numerical runs

For $x'=\lambda x$, both runs use **the same** $h$ and $\lambda$:

$$x_n=(1+h\lambda)^n x_0$$

$$\widetilde{x}_n=(1+h\lambda)^n\widetilde{x}_0.$$

Subtract the second equation from the first:

$$\begin{aligned}
\delta_n&=x_n-\widetilde{x}_n\\
&=(1+h\lambda)^n x_0-(1+h\lambda)^n\widetilde{x}_0\\
&=(1+h\lambda)^n(x_0-\widetilde{x}_0).
\end{aligned}$$

Therefore,

$$\boxed{\delta_n=(1+h\lambda)^n\delta_0.}$$

Taking magnitudes,

$$\boxed{|\delta_n|=|1+h\lambda|^n|\delta_0|.}$$

Call $R=1+h\lambda$ the **amplification factor**. It tells us how the initial difference changes after each step:

- $|R|<1$: the difference **decays** as $n$ increases.
- $|R|=1$: its magnitude **stays constant**.
- $|R|>1$: its magnitude **grows** with $n$.

For a mode that should decay, the **absolute stability** condition is $\boxed{|1+h\lambda|<1}$. If we merely require the perturbation to remain bounded for arbitrarily many steps in this scalar example, $|1+h\lambda|\le 1$ suffices.

**Important qualification:** This is a not a requirement for every arbitrary ODE. A physically growing solution may also amplify initial differences; on a fixed finite time interval, a numerical method can still be stable and convergent. The plots below explore the decaying-mode case.

## Example

Let's validate this result with an example. Let's consider the function is decaying, which means $\lambda=-2<0$ and apply the time stepping with a positive step size of $ h=0.25$ for $12$ steps. 

**Calculate whether this configuration results in a stable approximation. What is the amplification factor?**  

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (8, 4.3), "axes.grid": True, "grid.alpha": 0.25})

def euler_trajectory(lam, h, x0, steps):
    n = np.arange(steps + 1)
    return n * h, x0 * (1 + h * lam) ** n

lam, h, steps = -2.0, 0.25, 12
t, x = euler_trajectory(lam, h, 1.0, steps)
t_exact = np.linspace(0, t[-1], 100)
plt.plot(t_exact, np.exp(lam*t_exact), label="Exact solution", linewidth=2)
plt.plot(t, x, "o--", label="Explicit Euler")
plt.xlabel("Time"); plt.ylabel("x"); plt.title("$\lambda=-2$")
plt.legend(); plt.show()

As seen in the result, the numerical approximation can approximate the solution profile in general. There is a slight deviation which can be reduced by refining our discretization by using smaller step *(Test by yourself)*.

## 5. Visual experiment: do two nearby numerical runs stay close?

For $\lambda=-2$, change $h$ and compare two runs starting at $x_0=1$ and $\widetilde{x}_0=1.01$. Their initial difference is $|\delta_0|=0.01$.

In [ ]:
lam = -2.0
x0, x0_perturbed = 1.0, 1.01
fig, axs = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, h in zip(axs, [0.25, 0.75, 1.25]):
    n = np.arange(13)
    R = 1 + h*lam
    d = (x0_perturbed-x0)*R**n
    ax.plot(n, np.abs(d), "o-", label="|perturbation|")
    ax.set_title(f"h={h}, R={R:+.2f}, |R|={abs(R):.2f}")
    ax.set_xlabel("Step n"); ax.set_ylabel("|δₙ|")
    ax.legend()
fig.suptitle("Same ODE, different time steps", y=1.04)
plt.tight_layout(); plt.show()

As seen in the plots, when the absolute value of the amplification factor is smaller than $1$. The numerical scheme converges to the same solution. Otherwise, it amplifies the error and diverges. 

Change $h$ and watch the initial difference evolve. A negative $R$ causes alternating signs in the difference; only its magnitude determines whether the perturbation grows or decays.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

def show_perturbation(h=0.25):
    n = np.arange(16)
    R = 1 - 2 * h
    d = 0.01 * R**n

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.axhline(0, color="gray", linewidth=1)
    ax.plot(n, d, "o-")
    ax.set(
        xlabel="Step n",
        ylabel="Signed difference δₙ",
        title=f"R = {R:.3f}, |R| = {abs(R):.3f}"
    )
    ax.grid(True, alpha=0.3)
    plt.show()
    plt.close(fig)

slider = widgets.FloatSlider(
    value=0.25,
    min=0.05,
    max=1.5,
    step=0.05,
    description="Step size h",
    continuous_update=False
)

output = widgets.Output()

def update(change=None):
    with output:
        clear_output(wait=True)
        show_perturbation(slider.value)

slider.observe(update, names="value")

display(widgets.VBox([slider, output]))
update()

## 6. What other possibilities are there? 

Consider the constraint on the amplification factor again. What else satisfies this condition?

So far, we have considered a real, negative value of $\lambda$. But what happens if $\lambda$ is **complex**?

Our stability condition is still valid:

$$
\boxed{|1+h\lambda|<1}
$$

But how can $\lambda$ be complex?

Consider

$$
\lambda = a+ib, \qquad i=\sqrt{-1}.
$$

The analytical solution of our ODE is still

$$
x(t)=x_0e^{\lambda t}.
$$

Substituting the complex value of $\lambda$ gives

$$
x(t)=x_0e^{(a+ib)t}
=x_0e^{at}\left(\cos(bt)+i\sin(bt)\right).
$$

This means that the solution can have **oscillatory behavior**, while its amplitude can grow or decay depending on the real part of $\lambda$.

- $a<0$: The amplitude decays over time.
- $a=0$: The amplitude remains constant.
- $a>0$: The amplitude grows over time.
- $b\neq0$: The solution oscillates.

For example, $\lambda=-1+4i$ describes a decaying oscillation.

Although physical solutions are often real-valued, complex values naturally arise as eigenvalues of systems of real differential equations. We will return to this when considering systems of ODEs.


Let's explore how the solution looks like. Recall that for $\lambda=a+ib$, the analytical solution is

$$
x(t)=x_0e^{at}\left(\cos(bt)+i\sin(bt)\right).
$$

In the following interactive example, we can change $a$ and $b$ independently and observe how the solution behaves.

- **Real part ($a$):** Controls whether the amplitude grows, decays, or remains constant.
- **Imaginary part ($b$):** Controls the frequency of oscillations. If $b=0$, there are no oscillations.

The plot shows the real and imaginary parts of the solution, together with the amplitude envelope $\pm e^{at}$ (for $x_0=1$).

Try setting $a=0$ and changing $b$. Then keep $b$ fixed and change $a$. What happens to the solution in each case?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def show_complex_solution(a=-0.5, b=4.0):
    t = np.linspace(0, 10, 1000)
    x0 = 1.0

    x = x0 * np.exp((a + 1j*b) * t)
    envelope = np.exp(a*t)

    fig, ax = plt.subplots(figsize=(9, 4))

    ax.plot(t, x.real, label=r"$\mathrm{Re}(x(t))$")
    ax.plot(t, x.imag, label=r"$\mathrm{Im}(x(t))$", linestyle="--")
    ax.plot(t, envelope, "k:", alpha=0.6, label=r"$\pm e^{at}$")
    ax.plot(t, -envelope, "k:", alpha=0.6)

    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set(
        xlabel="Time t",
        ylabel="x(t)",
        title=fr"$\lambda = {a:.2f} + ({b:.2f})i$"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.show()
    plt.close(fig)

widgets.interact(
    show_complex_solution,
    a=widgets.FloatSlider(
        value=-0.5, min=-1.0, max=0.5, step=0.1,
        description="Real (a)", continuous_update=False
    ),
    b=widgets.FloatSlider(
        value=4.0, min=0.0, max=10.0, step=0.5,
        description="Imag (b)", continuous_update=False
    )
)

## 7. What does this mean for numerical stability?

For explicit Euler, we still have

$$
R=1+h\lambda.
$$

With $\lambda=a+ib$,

$$
R=(1+ha)+ihb.
$$

Therefore, the stability condition becomes

$$
\boxed{(1+ha)^2+(hb)^2<1.}
$$

**This describes a circle centered at $-1$ with radius $1$ in the complex $h\lambda$-plane.**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def show_euler_stability(a=-0.5, b=4.0, h=0.2):
    lam = a + 1j * b
    z = h * lam
    R = 1 + z

    # Exact and numerical solutions
    t_end = 10.0
    t_exact = np.linspace(0, t_end, 1000)
    x_exact = np.exp(lam * t_exact)

    n = np.arange(int(t_end / h) + 1)
    t_euler = n * h
    x_euler = R**n

    with plt.ioff():
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

    # Left: Stability region
    theta = np.linspace(0, 2 * np.pi, 400)
    xc = -1 + np.cos(theta)
    yc = np.sin(theta)

    ax1.fill(xc, yc, color="lightblue", alpha=0.5, label="Stable region")
    ax1.plot(xc, yc, color="blue", label=r"$|1+h\lambda|=1$")
    ax1.plot(z.real, z.imag, "ro", markersize=9, label=r"$h\lambda$")

    ax1.axhline(0, color="gray", linewidth=0.8)
    ax1.axvline(0, color="gray", linewidth=0.8)
    ax1.set(
        xlabel=r"$\mathrm{Re}(h\lambda)$",
        ylabel=r"$\mathrm{Im}(h\lambda)$",
        title="Explicit Euler stability region",
        xlim=(-3, 1),
        ylim=(-2, 2)
    )
    ax1.set_aspect("equal")
    ax1.legend(loc="upper left", fontsize=9)
    ax1.grid(True, alpha=0.2)

    # Right: Exact vs Euler
    ax2.plot(t_exact, x_exact.real, label="Exact", linewidth=2)
    ax2.plot(t_euler, x_euler.real, "o--", markersize=3, label="Explicit Euler")

    ax2.axhline(0, color="gray", linewidth=0.8)
    ax2.set(
        xlabel="Time t",
        ylabel=r"$\mathrm{Re}(x(t))$",
        title=f"|1 + hλ| = {abs(R):.3f} "
              + ("(Stable)" if abs(R) < 1 else "(Unstable)")
    )
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.suptitle(
        f"λ = {a:.2f} + ({b:.2f})i,   h = {h:.2f}",
        fontsize=12
    )
    fig.tight_layout()

    display(fig)
    plt.close(fig)


controls = {
    "a": widgets.FloatSlider(
        value=-0.5, min=-2.0, max=1.0, step=0.1,
        description="Real (a)", continuous_update=False
    ),
    "b": widgets.FloatSlider(
        value=4.0, min=0.0, max=10.0, step=0.5,
        description="Imag (b)", continuous_update=False
    ),
    "h": widgets.FloatSlider(
        value=0.2, min=0.05, max=1.0, step=0.05,
        description="Step size h", continuous_update=False
    )
}

out = widgets.interactive_output(show_euler_stability, controls)
display(widgets.VBox([*controls.values(), out]))

### Stability does not necessarily mean accuracy

Try setting $a=-0.7$, $b=1$, and $h=0.5$ in the interactive example.

You may notice that **explicit Euler oscillates faster than the exact solution**, even though both solutions decay over time. Why?

Recall that the exact solution is

$$
x(t)=e^{at}\left(\cos(bt)+i\sin(bt)\right).
$$

The **analytical angular frequency** is the frequency of the exact solution:

$$
\boxed{\omega_{\mathrm{exact}}=|b|=1.}
$$

For explicit Euler, the amplification factor is

$$
R=1+h\lambda=0.65+0.5i.
$$

Since $R$ is complex, we can express it in polar form:

$$
R=|R|e^{i\theta},
\qquad \theta=\arg(R).
$$

After $n$ steps,

$$
x_n=R^n x_0=|R|^n e^{in\theta}x_0.
$$

Each numerical step advances the oscillation by an angle $\theta$. Since each step takes $h$ time units, the **numerical angular frequency** is

$$
\boxed{
\omega_{\mathrm{Euler}}=\frac{|\theta|}{h}
=\frac{|\arg(1+h\lambda)|}{h}.
}
$$

For our example,

$$
\omega_{\mathrm{Euler}}
=\frac{\arg(0.65+0.5i)}{0.5}
\approx1.312.
$$

So explicit Euler oscillates approximately **31% faster** than the exact solution.

How can we reduce this error?

One possibility is to **decrease the step size $h$**. As $h$ becomes smaller, the numerical solution approaches the exact solution, and the numerical frequency approaches the analytical frequency.

However, smaller step sizes also mean more computational steps and therefore potentially longer runtimes.

This illustrates two different numerical errors:

- **Amplitude error:** The numerical solution grows or decays at a different rate.
- **Phase error:** The numerical solution oscillates at a different frequency.

**Try it yourself:**

1. Set $a=-0.7$, $b=1$, and $h=0.5$. Compare the oscillations.
2. Decrease $h$ to $0.2$, then $0.1$, and finally $0.05$. Does the numerical solution approach the exact solution?
3. Set $a=0$, $b=4$, and $h=0.2$. What happens to the amplitude of the numerical solution? Is explicit Euler stable? What happens when you change $h$?
4. Keep $a=-1$ and $b=4$. Increase $h$ from $0.05$ to $0.2$. At what point does the numerical solution become unstable?

**Takeaway:** Numerical stability does not guarantee numerical accuracy. Even when a numerical solution is stable, it can have significant amplitude and phase errors. Reducing the step size generally improves accuracy, but increases computational cost.

## 8. Numerical stability for systems of ODEs

So far, we have considered a single differential equation:

$$
x'(t)=\lambda x(t).
$$

But what happens if we have multiple equations that depend on each other?

Consider a system of two coupled ODEs:

$$
\begin{aligned}
x'(t)&=-2x(t)+y(t),\\
y'(t)&=x(t)-3y(t).
\end{aligned}
$$

Here, the change in $x$ depends on both $x$ and $y$, and similarly for $y$.

We can write this system in matrix form:

$$
\boxed{\mathbf{x}'(t)=A\mathbf{x}(t)}
$$

where

$$
\mathbf{x}(t)=
\begin{bmatrix}
x(t)\\
y(t)
\end{bmatrix},
\qquad
A=
\begin{bmatrix}
-2&1\\
1&-3
\end{bmatrix}.
$$

### Applying explicit Euler

Recall that for a single equation, explicit Euler gives

$$
x_{n+1}=(1+h\lambda)x_n.
$$

For a system, we apply the same idea:

$$
\mathbf{x}_{n+1}=\mathbf{x}_n+hA\mathbf{x}_n.
$$

Rearranging,

$$
\boxed{\mathbf{x}_{n+1}=(I+hA)\mathbf{x}_n}
$$

where $I$ is the identity matrix.

The scalar amplification factor $R=1+h\lambda$ has now become an **amplification matrix**:

$$
\boxed{R=I+hA.}
$$

After $n$ steps, we obtain

$$
\mathbf{x}_n=R^n\mathbf{x}_0.
$$

### What about stability?

As before, consider two solutions starting from slightly different initial conditions. Their difference satisfies

$$
\boxed{\boldsymbol{\delta}_n=R^n\boldsymbol{\delta}_0.}
$$

We want to know whether this perturbation grows or decays over time.

For a single equation, we checked

$$
|1+h\lambda|<1.
$$

But how can we check this for a matrix?

This is where **eigenvalues** become useful.

If $A$ has eigenvalues $\lambda_1,\lambda_2,\ldots,\lambda_m$, the amplification matrix $R=I+hA$ has eigenvalues

$$
1+h\lambda_1,\quad 1+h\lambda_2,\quad\ldots,\quad 1+h\lambda_m.
$$

Each eigenvalue represents a mode of the system that behaves similarly to our single-equation example.

Therefore, for asymptotic stability, we require

$$
\boxed{|1+h\lambda_i|<1\qquad\text{for all }i.}
$$

In other words, **all modes must be stable**. If even one mode is unstable, a perturbation in that direction can grow.

Instead of checking each eigenvalue individually, we can use the **spectral radius** of the amplification matrix $R$.

The spectral radius is simply the largest absolute value among the eigenvalues of a matrix:

$$
\boxed{\rho(R)=\max_i|\mu_i|}
$$

where $\mu_i$ are the eigenvalues of $R$.

Since $R=I+hA$, we have $\mu_i=1+h\lambda_i$. Therefore, our stability condition becomes

$$
\boxed{\rho(I+hA)<1.}
$$

This means that **the largest amplification factor must be smaller than 1**. If this condition is satisfied, all perturbations eventually decay. For some matrices, perturbations may temporarily grow before decaying.

### Applying this to our example

For our example, the eigenvalues of $A$ are approximately

$$
\lambda_1=-1.382,\qquad \lambda_2=-3.618.
$$

Both are negative, so the exact solution decays. However, explicit Euler still requires a sufficiently small step size:

$$
h<\min\left(\frac{2}{1.382},\frac{2}{3.618}\right)
\approx0.553.
$$

Let's compare different step sizes:

| Step size $h$ | $\rho(I+hA)$ | Stable? |
|---|---|---|
| $0.25$ | $0.655$ | Yes |
| $0.50$ | $0.809$ | Yes |
| $0.60$ | $1.171$ | No |

**An important observation:** The eigenvalue with the largest magnitude determines the maximum stable step size in this example.

Let's explore what happens when we change the step size and observe the spectral radius.

### Try it yourself

Change the step size $h$ and observe how the numerical solution behaves.

1. Start with $h=0.25$. Does explicit Euler approximate the exact solution well?
2. Increase to $h=0.5$. Is the numerical solution still stable? Is it accurate?
3. Increase to $h=0.6$. What happens? Why does the numerical solution start oscillating or growing even though the exact solution decays?
4. Decrease $h$ again. What happens to the accuracy?

**Think about:** Why does the system become unstable even though both eigenvalues of $A$ are negative?

**Takeaway:** For a system of ODEs, numerical stability depends on all eigenvalues. Even if the exact solution is stable, the numerical method can become unstable if the step size is too large.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.linalg import expm

A = np.array([
    [-2.0, 1.0],
    [ 1.0,-3.0]
])

x0 = np.array([1.0, 0.5])

def show_system_stability(h=0.25):
    t_end = 10.0
    n_steps = int(np.ceil(t_end / h))
    t_euler = h * np.arange(n_steps + 1)

    R = np.eye(2) + h * A
    eig = np.linalg.eigvals(R)
    rho = max(abs(eig))

    x_euler = np.zeros((n_steps + 1, 2))
    x_euler[0] = x0

    for n in range(n_steps):
        x_euler[n + 1] = R @ x_euler[n]

    t_exact = np.linspace(0, t_end, 500)
    x_exact = np.array([expm(A * t) @ x0 for t in t_exact])

    fig, ax = plt.subplots(figsize=(9, 4.5))

    ax.plot(t_exact, x_exact[:, 0], label="Exact x", color="C0")
    ax.plot(t_exact, x_exact[:, 1], label="Exact y", color="C1")

    ax.plot(t_euler, x_euler[:, 0], "o--",
            color="C0", markersize=3, label="Euler x")
    ax.plot(t_euler, x_euler[:, 1], "s--",
            color="C1", markersize=3, label="Euler y")

    ax.axhline(0, color="gray", linewidth=0.8)
    ax.set(
        xlabel="Time t",
        ylabel="Solution",
        title=f"h = {h:.2f}, spectral radius = {rho:.3f}"
    )
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.show()
    plt.close(fig)

widgets.interact(
    show_system_stability,
    h=widgets.FloatSlider(
        value=0.25,
        min=0.05,
        max=1.0,
        step=0.05,
        description="Step size h",
        continuous_update=False
    )
)

## 9. What if the eigenvalues are complex?

So far, our system had two real eigenvalues. But as we saw for a single ODE, eigenvalues can also be complex.

Consider another system:

$$
\begin{aligned}
x'(t)&=-x(t)-4y(t),\\
y'(t)&=4x(t)-y(t).
\end{aligned}
$$

In matrix form,

$$
\mathbf{x}'(t)=A\mathbf{x}(t),
\qquad
A=
\begin{bmatrix}
-1&-4\\
4&-1
\end{bmatrix}.
$$

Although the matrix contains only real numbers, its eigenvalues are

$$
\boxed{\lambda_{1,2}=-1\pm4i.}
$$

As before, the real part determines the decay or growth, while the imaginary part determines the oscillatory behavior.

For explicit Euler, the amplification matrix is

$$
R=I+hA.
$$

Its eigenvalues are

$$
\mu_{1,2}=1+h\lambda_{1,2}
=1-h\pm4hi.
$$

The spectral radius is therefore

$$
\boxed{
\rho(R)=\max_i|\mu_i|
=\sqrt{(1-h)^2+(4h)^2}.
}
$$

The stability condition remains

$$
\boxed{\rho(R)<1.}
$$

Let's visualize this in the **complex plane**.

We will plot the eigenvalues of $R$, the unit circle representing the stability boundary, and a circle with radius $\rho(R)$.

- If the spectral radius is smaller than $1$, all eigenvalues lie inside the unit circle.
- If the spectral radius is larger than $1$, at least one eigenvalue lies outside, and the numerical solution is unstable.
- Complex eigenvalues can introduce oscillations, even when the numerical solution is stable.

Change the step size and observe how the eigenvalues move.

### Try it yourself

1. Start with $h=0.1$. Where are the eigenvalues relative to the unit circle? Does the numerical solution decay?
2. Increase the step size to $h=0.12$. What happens to the spectral radius?
3. Increase it further to $h=0.2$. What happens to the numerical solution?
4. Can you determine the maximum stable step size from the eigenvalues?

For this system, the stability condition becomes

$$
(1-h)^2+16h^2<1.
$$

Simplifying,

$$
\boxed{0<h<\frac{2}{17}\approx0.118.}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

A = np.array([
    [-1.0, -4.0],
    [ 4.0, -1.0]
])

def show_complex_system(h=0.1):
    R = np.eye(2) + h * A
    eig = np.linalg.eigvals(R)
    rho = np.max(np.abs(eig))

    theta = np.linspace(0, 2*np.pi, 400)

    with plt.ioff():
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    # Left: eigenvalues in complex plane
    ax1.plot(np.cos(theta), np.sin(theta),
             "k--", label="Unit circle (stability boundary)")

    ax1.plot(rho*np.cos(theta), rho*np.sin(theta),
             color="C1", label=fr"Spectral radius $\rho={rho:.3f}$")

    ax1.scatter(eig.real, eig.imag, color="C3", s=75,
                zorder=3, label="Eigenvalues of R")

    ax1.axhline(0, color="gray", linewidth=0.8)
    ax1.axvline(0, color="gray", linewidth=0.8)
    ax1.set(
        xlabel=r"$\mathrm{Re}(\mu)$",
        ylabel=r"$\mathrm{Im}(\mu)$",
        title="Eigenvalues of amplification matrix",
        xlim=(-1.6, 1.6),
        ylim=(-1.6, 1.6)
    )
    ax1.set_aspect("equal")
    ax1.grid(True, alpha=0.2)
    ax1.legend(fontsize=8)

    # Right: numerical solution
    t_end = 5.0
    n_steps = int(t_end / h)
    t = np.arange(n_steps + 1) * h

    x = np.zeros((n_steps + 1, 2))
    x[0] = [1.0, 0.0]

    for n in range(n_steps):
        x[n+1] = R @ x[n]

    ax2.plot(t, x[:, 0], "o-", markersize=3, label="x(t)")
    ax2.plot(t, x[:, 1], "s-", markersize=3, label="y(t)")
    ax2.axhline(0, color="gray", linewidth=0.8)

    ax2.set(
        xlabel="Time t",
        ylabel="Numerical solution",
        title=f"Spectral radius = {rho:.3f} "
              + ("(Stable)" if rho < 1 else "(Unstable)")
    )
    ax2.grid(True, alpha=0.3)
    ax2.legend()

    fig.tight_layout()
    plt.show()
    plt.close(fig)

widgets.interact(
    show_complex_system,
    h=widgets.FloatSlider(
        value=0.1,
        min=0.02,
        max=0.3,
        step=0.02,
        description="Step size h",
        continuous_update=False
    )
)

## 10. Explicit vs. implicit Euler

So far, we have only considered **explicit Euler**. But what happens if we evaluate the derivative at the next time step instead of the current one?

Recall that explicit Euler gives

$$
\mathbf{x}_{n+1}=\mathbf{x}_n+hA\mathbf{x}_n.
$$

For **implicit Euler**, we instead use

$$
\mathbf{x}_{n+1}=\mathbf{x}_n+hA\mathbf{x}_{n+1}.
$$

Notice that the unknown $\mathbf{x}_{n+1}$ appears on both sides. Rearranging,

$$
(I-hA)\mathbf{x}_{n+1}=\mathbf{x}_n.
$$

Therefore,

$$
\boxed{\mathbf{x}_{n+1}=(I-hA)^{-1}\mathbf{x}_n.}
$$


### What about stability?

Both methods have an amplification matrix, but they are different:

| | Explicit Euler | Implicit Euler |
|---|---|---|
| Amplification matrix | $R_E=I+hA$ | $R_I=(I-hA)^{-1}$ |
| Eigenvalue amplification | $1+h\lambda$ | $\frac{1}{1-h\lambda}$ |
| Stability condition | $\rho(R_E)<1$ | $\rho(R_I)<1$ |

For a single eigenvalue $\lambda$, the stability conditions become

**Explicit Euler:**

$$
|1+h\lambda|<1.
$$

**Implicit Euler:**

$$
\left|\frac{1}{1-h\lambda}\right|<1.
$$

See the stability regions for these two methods. 

In [ ]:
X,Y=np.meshgrid(np.linspace(-3,2,450),np.linspace(-2.5,2.5,450))
Z=X+1j*Y
fig,axs=plt.subplots(1,2,figsize=(12,5))
for ax,func,title in [(axs[0],lambda z:1+z,"Explicit Euler"),
                       (axs[1],lambda z:1/(1-z),"Implicit Euler")]:
    amp=np.abs(func(Z))
    ax.contourf(X,Y,amp,levels=[0,1,1e9],colors=["#d5eee3","#f4dddd"],alpha=0.9)
    ax.contour(X,Y,amp,levels=[1],colors=["#157a5b"],linewidths=2)
    ax.axhline(0,color="gray",linewidth=0.8);ax.axvline(0,color="gray",linewidth=0.8)
    ax.set(xlabel="Re(z)",ylabel="Im(z)",title=title)
    ax.set_aspect("equal")
plt.tight_layout();plt.show()

An important difference is that **implicit Euler is stable for any positive step size when all eigenvalues have negative real parts**. This property is called *A-stability*.

However, does this mean that implicit Euler always produces an accurate solution?

Let's investigate this using an oscillating system.

### Example: Harmonic oscillator

Consider a simple harmonic oscillator, such as a mass attached to a spring.

Its governing equation is

$$
x''(t)=-\omega^2x(t),
$$

where $x$ is the position and $\omega$ is the angular frequency.

By introducing the velocity $v=x'$, we can rewrite this second-order equation as a system of first-order ODEs:

$$
\begin{aligned}
x' &= v,\\
v' &= -\omega^2x.
\end{aligned}
$$

For $\omega=1$, the matrix form becomes

$$
\begin{bmatrix}
x'\\
v'
\end{bmatrix}
=
\underbrace{
\begin{bmatrix}
0&1\\
-1&0
\end{bmatrix}
}_{A}
\begin{bmatrix}
x\\
v
\end{bmatrix}.
$$

The eigenvalues of $A$ are

$$
\boxed{\lambda_{1,2}=\pm i.}
$$

These purely imaginary eigenvalues describe oscillations with constant amplitude.

For the initial conditions $x(0)=1$ and $v(0)=0$, the analytical solution is

$$
x(t)=\cos(t),\qquad v(t)=-\sin(t).
$$

We can visualize the solution in a **phase portrait**, where we plot velocity against position instead of time.

For the exact solution,

$$
x^2+v^2=1.
$$

This describes a circle: the system continuously oscillates without gaining or losing energy.

The total energy is

$$
\boxed{E=\frac{1}{2}(x^2+v^2).}
$$

For the exact solution, this energy remains constant.

**What happens if we solve the same system using explicit and implicit Euler?**

Let's compare their trajectories and energy evolution.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

def compare_euler(h=0.2):
    A = np.array([[0., 1.],
                  [-1., 0.]])
    I = np.eye(2)

    R_exp = I + h * A
    R_imp = np.linalg.solve(I - h * A, I)

    t_end = 15
    n = int(t_end / h)
    t = np.arange(n + 1) * h

    initial = np.array([1., 0.])
    explicit = np.zeros((n + 1, 2))
    implicit = np.zeros((n + 1, 2))
    explicit[0] = initial
    implicit[0] = initial

    for k in range(n):
        explicit[k + 1] = R_exp @ explicit[k]
        implicit[k + 1] = R_imp @ implicit[k]

    exact = np.column_stack((np.cos(t), -np.sin(t)))

    E_exp = 0.5 * np.sum(explicit**2, axis=1)
    E_imp = 0.5 * np.sum(implicit**2, axis=1)

    with plt.ioff():
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

    # Phase portrait
    ax1.plot(exact[:, 0], exact[:, 1], "k-", label="Exact")
    ax1.plot(explicit[:, 0], explicit[:, 1], label="Explicit Euler")
    ax1.plot(implicit[:, 0], implicit[:, 1], label="Implicit Euler")

    ax1.set(
        xlabel="Position x",
        ylabel="Velocity v",
        title="Phase portrait"
    )
    ax1.set_aspect("equal", adjustable="datalim")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Energy evolution
    ax2.axhline(0.5, color="black", label="Exact energy")
    ax2.plot(t, E_exp, label="Explicit Euler")
    ax2.plot(t, E_imp, label="Implicit Euler")

    ax2.set(
        xlabel="Time t",
        ylabel="Energy",
        title="Energy evolution"
    )
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()
    plt.close(fig)

widgets.interact(
    compare_euler,
    h=widgets.FloatSlider(
        value=0.2,
        min=0.05,
        max=0.5,
        step=0.05,
        description="Step size h",
        continuous_update=False
    )
)


### What do we observe?

The exact solution follows a closed orbit in the phase portrait, meaning its energy remains constant.

However, the two numerical methods behave differently:

- **Explicit Euler:** The trajectory spirals outward. The numerical solution gains artificial energy, and its amplitude increases.
- **Implicit Euler:** The trajectory spirals inward. The numerical solution loses artificial energy, and its amplitude decreases.

Why does this happen?

Recall that the eigenvalues of our system are $\lambda=\pm i$.

For explicit Euler, the amplification factors are

$$
\mu_E=1\pm hi,
$$

with magnitude

$$
|\mu_E|=\sqrt{1+h^2}>1.
$$

For implicit Euler, the amplification factors are

$$
\mu_I=\frac{1}{1\mp hi},
$$

with magnitude

$$
|\mu_I|=\frac{1}{\sqrt{1+h^2}}<1.
$$

Therefore, explicit Euler amplifies the oscillations, while implicit Euler damps them.

### Try it yourself

1. Start with $h=0.2$. Compare the trajectories of both methods with the exact solution.
2. Increase the step size to $h=0.5$. What happens to the energy?
3. Decrease the step size to $h=0.05$. Do both numerical solutions approach the exact solution?
4. Which method is more stable? Does that necessarily make it more accurate?

**Takeaway:** Stability and accuracy are not the same. Implicit Euler avoids the artificial growth produced by explicit Euler, but it introduces artificial damping. Neither method perfectly preserves the energy of this oscillating system.

This also shows why the choice of numerical method matters, not just the step size.

## Summary

- **Numerical stability** describes how small perturbations affect numerical solutions.
- **The amplification factor** determines how perturbations evolve: $|R|<1$ gives decay, $|R|=1$ preserves their magnitude, and $|R|>1$ gives growth for the scalar test equation.
- **Complex eigenvalues** describe oscillatory modes. Their real parts determine growth or decay, while their imaginary parts determine oscillation frequency.
- **For coupled linear systems**, examine every eigenvalue: each scaled eigenvalue $h\lambda$ must lie within the method's stability region for all modes to decay. This can be done by analyzing the maximum eigenvalue. 
- **Stability does not guarantee accuracy.** A stable method can still distort oscillation frequency or amplitude.
- **For the harmonic oscillator**, explicit Euler introduces artificial growth, while implicit Euler introduces artificial damping.
- **Both method and step size matter.** Choose them to control numerical instability while resolving the behavior of interest.

**Reference:** [David Eberly, *Stability Analysis for Systems of Differential Equations*](https://www.geometrictools.com/Documentation/StabilityAnalysis.pdf)